# GameTheory-10 — Équilibres Parfaits de Sous-Jeux et Induction Avant (Twin C#)
**Navigation** : [<< 9-BackwardInduction (twin C#)](GameTheory-09-BackwardInduction-CSharp.ipynb) | [Twin Python](GameTheory-10-ForwardInduction-SPE-Python.ipynb) | [Index](README.md) | [11-BayesianGames (twin C#) >>](GameTheory-11-BayesianGames-CSharp.ipynb)


**Jumeau C# (.NET Interactive)** de `GameTheory-10-ForwardInduction-SPE-Python.ipynb` — parité .NET ⇄ Python.

Ce notebook réimplémente from-scratch en C# pur (BCL .NET 9, **0 NuGet**) les **raffinements d'équilibre** sur la forme extensive : équilibre parfait de sous-jeux (SPE via **backward induction**), analyse des **menaces crédibles**, **perfection en mains tremblantes** (trembling-hand), **induction avant** (forward induction, logique de Cho-Kreps) et **signal par brûlage** (burn money).

> L'original Python s'appuie sur `numpy`/`matplotlib` pour les calculs et graphes ; ce twin traduit les algorithmes en type system C# + `HashSet`/LINQ, et restitue les résultats en **tables console**. La valeur ajoutée n'est pas un miroir : c'est un **moteur de théorie des jeux extensifs typé**, où l'arbre de jeu, la récursion de backward induction et la détection des sous-jeux vivent dans le système de types.

**Voir aussi** : [GameTheory-10 (Python)](GameTheory-10-ForwardInduction-SPE-Python.ipynb).
### Ce que ce carnet calcule, en une ligne chacun

| Section | Exemple traité | Résultat calculé ici |
|---|---|---|
| §1 | Jeu d'entrée (entrant vs incumbent) | SPE **(In, Acc)**, paiement **(2, 1)** |
| §1 | Validation croisée Gambit | `gambit-lcp` = 1 équilibre, `gambit-enumpure` = 2 Nash purs |
| §2 | Engagement (Fight rapporte 3) | la menace devient crédible, le SPE **bascule vers Out (1, 2)** |
| §3 | Stag Hunt + option extérieure de 3,5 | seul Stag peut dépasser l'option ; l'inférence dépend de la rationalité stricte et des croyances |
| §4 | Tremblement ε = 0,1 | espérance **1,900 > 1** : le SPE reste parfait |
| §5 | Burn money (c = 0,5) sur BoS | seul Opéra peut dépasser la continuation Football sans brûlage ; comparaison conditionnelle, pas sélection de tous les équilibres |

Tout est calculé **from scratch en C# pur (BCL .NET 9, 0 NuGet)** : l'arbre, la récursion de backward induction, les sous-jeux, la perturbation ε et la logique de signal. Seule la §1 ajoute une **validation croisée** par un solveur externe indépendant (Gambit), pour que le SPE trouvé à la main ne soit jamais la seule voix du carnet.

## Références

**Osborne & Rubinstein (1994)**, §6.2 « Subgame Perfect Equilibrium » puis ch. 6 pp. 110-114 (induction avant, cf. index) : les deux lectures de la rationalite sequentielle. Condense : **Leyton-Brown & Shoham (2008)**, ch. 4. Gisement cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`.


## Objectifs d'apprentissage

1. Définir un **jeu sous forme extensive** (arbre, nœuds de décision, nœuds terminaux, paiements).
2. Calculer l'**équilibre parfait de sous-jeux (SPE)** par **backward induction** (récursion sur l'arbre).
3. Distinguer **menace crédible** vs **non-crédible** (pourquoi certains équilibres de Nash ne survivent pas).
4. Appliquer la **perfection en mains tremblantes** (perturbation ε).
5. Maîtriser l'**induction avant** (forward induction) : déduire les intentions d'un joueur de ses actions hors-chemin.
6. Analyser le **signal par brûlage** (burn money).

### Prérequis
- Théorie de l'équilibre de Nash (cf. [GameTheory-4](GameTheory-04-NashEquilibrium-CSharp.ipynb)).
- Forme extensive (cf. [GameTheory-7](GameTheory-07-ExtensiveForm-CSharp.ipynb)).
### Comment lire ce carnet

Chaque section suit le même triptyque : **un concept défini en une cellule de prose → un mini-programme C# auto-contenu qui le calcule → la lecture du résultat en face des nombres produits**. Les cellules de code ne dépendent que de l'API de la cellule 3 (setup) : on peut lire les sections 2 à 6 dans n'importe quel ordre après la §1. Les exercices de la §7 réutilisent exactement la même API — compléter un exercice n'exige jamais d'import supplémentaire.

In [1]:
// Setup : types de base pour un jeu sous forme extensive.
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;

// Helper d'affichage (headless-safe : .Display() capture text/plain).
public static void Show(object? o) => (o?.ToString() ?? "<null>").Display();

// Noeud d'un arbre de jeu extensif (information parfaite : singleton info-set).
public sealed class GameNode
{
    public int Id { get; }
    public int Player { get; }               // joueur qui decide (-1 = terminal)
    public List<GameNode> Children { get; } = new();
    public List<string> Actions { get; } = new();   // etiquette d'action par enfant
    public double[]? Payoffs { get; }        // paiement terminal (null si decision)
    public bool IsTerminal => Player < 0;
    public GameNode(int id, int player, double[]? payoffs = null)
    { Id = id; Player = player; Payoffs = payoffs; }
    public GameNode AddChild(string action, GameNode child) { Actions.Add(action); Children.Add(child); return this; }
}

// Un jeu extensif = racine + nombre de joueurs.
public sealed class ExtensiveFormGame
{
    public GameNode Root { get; }
    public int NPlayers { get; }
    public string[] PlayerNames { get; }
    public ExtensiveFormGame(GameNode root, int n, string[]? names = null)
    { Root = root; NPlayers = n; PlayerNames = names ?? Enumerable.Range(0,n).Select(i=>$"J{i+1}").ToArray(); }
}

Show("Types charges : GameNode (arbre), ExtensiveFormGame, helper Show().");


Types charges : GameNode (arbre), ExtensiveFormGame, helper Show().

### Lecture : l'API que le carnet construit (cellule ci-dessus)

La sortie confirme les trois briques : `GameNode` (un nœud : joueur, paiement terminal, ou enfants étiquetés par action), `ExtensiveFormGame` (l'arbre + les joueurs) et le helper `Show()` qui imprimera les arbres des sections suivantes. Tout le carnet — exercices compris — ne construira rien d'autre : un raffinement d'équilibre n'est jamais qu'une **façon de parcourir cet arbre**. Le `#nullable enable` ouvre chaque cellule de code : ce jumeau ne dépend d'aucun paquet, tout vit dans la BCL .NET 9.

## 1. Sous-jeux et équilibre parfait (SPE)

### 1.1 Définition d'un sous-jeu
Un **sous-jeu** est un sous-arbre commençant à un nœud de décision et contenant tous ses descendants (sans couper d'information). En information parfaite, **tout nœud de décision est racine d'un sous-jeu**.

### 1.2 SPE par backward induction
L'**équilibre parfait de sous-jeux** (Selten 1965) est un profil de stratégie qui forme un équilibre de Nash dans **chaque** sous-jeu. On le calcule par **backward induction** : à chaque nœud de décision, le joueur choisit l'action maximisant son paiement sachant que la suite se jouera selon le SPE des sous-jeux plus profonds.

> Le SPE élimine les **menaces non crédibles** : un équilibre de Nash peut reposer sur une action hors-chemin qu'un joueur rationnel ne jouerait jamais si le nœud était atteint. Le SPE, lui, exige la rationalité à **tous** les nœuds.
**Pourquoi l'algorithme termine** : l'arbre est fini et sans cycle, la récursion traite chaque nœud de décision **exactement une fois** (post-ordre : les enfants avant le parent) — le coût est **linéaire en la taille de l'arbre**, pas exponentiel. C'est la même asymétrie qui rend le SPE *calculable* là où énumérer tous les profils de stratégies serait explosif : le jeu d'entrée ci-dessous n'a que 2 sous-jeux-racines (les nœuds 1 et 2, visibles en sortie de la §1.3), mais un arbre de profondeur *d* avec 2 actions par nœud porterait déjà 2^d plans d'action par joueur.

In [2]:
// Backward induction : retourne les paiements SPE + le chemin d'actions.
#nullable enable
public static (double[] payoffs, List<(int nodeId, string action)> path) BackwardInduction(GameNode n)
{
    if (n.IsTerminal) return (n.Payoffs!, new List<(int,string)>());
    double[]? best = null;
    int bestIdx = 0;
    for (int i = 0; i < n.Children.Count; i++)
    {
        var (cp, _) = BackwardInduction(n.Children[i]);
        if (best == null || cp[n.Player] > best[n.Player] + 1e-12)
        { best = cp; bestIdx = i; }
    }
    var path = new List<(int,string)> { (n.Id, n.Actions[bestIdx]) };
    var (_, sub) = BackwardInduction(n.Children[bestIdx]);
    path.AddRange(sub);
    return (best!, path);
}

// Enumere tous les sous-jeux (racines = tous les nœuds de decision, info parfaite).
public static List<GameNode> AllDecisionNodes(GameNode n, List<GameNode>? acc = null)
{
    acc ??= new();
    if (!n.IsTerminal) acc.Add(n);
    foreach (var c in n.Children) AllDecisionNodes(c, acc);
    return acc;
}

Show("BackwardInduction + enumeration des sous-jeux definis.");


BackwardInduction + enumeration des sous-jeux definis.

### Interprétation : SPE sur le jeu d'entrée (entrant vs incumbent)

Le jeu d'entrée canonique : un **entrant** décide d'**Entrer** (In) ou **Rester dehors** (Out). S'il entre, l'**incumbent** choisit **Accommoder** (Acc) ou **Combattre** (Fight).

- `Out` → entrant 1, incumbent 2
- `In → Acc` → entrant 2, incumbent 1
- `In → Fight` → entrant −1, incumbent −1

L'incumbent a une **menace** (« je combattrai »), mais au nœud incumbent, Accommoder (1) domine strictement Combattre (−1) : la menace est **non crédible**.
**L'arithmétique de la crédibilité** : au nœud de l'incumbent, Accommoder rapporte **1** et Combattre **−1** — dévier vers Acc rapporte donc **+2**. Le paiement d'équilibre de l'entrant (**2**) n'est protégé par aucun contrat : il tient entièrement à la rationalité de l'incumbent **une fois le nœud atteint**. Remarquez aussi l'asymétrie des issues de Nash : (In, Acc) donne (2, 1) à (Entrant, Incumbent), (Out, Fight) donne (1, 2) — l'incumbent *préférerait* l'issue Out, mais ne peut pas la forcer : c'est précisément ce que la non-crédibilité signifie.

In [3]:
// Construction de l'arbre du jeu d'entree.
var nOut = new GameNode(3, -1, new double[]{ 1.0, 2.0 });
var nAcc = new GameNode(4, -1, new double[]{ 2.0, 1.0 });
var nFight = new GameNode(5, -1, new double[]{ -1.0, -1.0 });
var incumbent = new GameNode(2, 1);  // joueur 2 (incumbent) decide
incumbent.AddChild("Acc", nAcc).AddChild("Fight", nFight);
var entrant = new GameNode(1, 0);   // joueur 1 (entrant) decide
entrant.AddChild("Out", nOut).AddChild("In", incumbent);

var entryGame = new ExtensiveFormGame(entrant, 2, new[]{ "Entrant", "Incumbent" });

var (spePay, spePath) = BackwardInduction(entryGame.Root);
var subgames = AllDecisionNodes(entryGame.Root);

var sb = new System.Text.StringBuilder();
sb.AppendLine("Jeu d'entree (entrant vs incumbent)");
sb.AppendLine($"  Joueurs : {string.Join(", ", entryGame.PlayerNames)}");
sb.AppendLine($"  Sous-jeux (racines) : {subgames.Count} (nœuds {string.Join(", ", subgames.Select(s=>s.Id))})");
sb.AppendLine();
sb.AppendLine("SPE (backward induction) :");
foreach (var (nid, act) in spePath) sb.AppendLine($"  Nœud {nid} -> {act}");
sb.AppendLine($"  Paiement SPE = ({spePay[0]}, {spePay[1]})  (Entrant, Incumbent)");
sb.AppendLine();
sb.AppendLine("Analyse menace (Fight) : au nœud incumbent (joueur 1),");
sb.AppendLine($"  Acc = {nAcc.Payoffs![1]}, Fight = {nFight.Payoffs![1]}  -> Accommoder domine strictement.");
sb.AppendLine("  => 'Fight' est une menace NON CRÉDIBLE (hors-chemin SPE).");
Show(sb.ToString());


Jeu d'entree (entrant vs incumbent)
  Joueurs : Entrant, Incumbent
  Sous-jeux (racines) : 2 (nœuds 1, 2)

SPE (backward induction) :
  Nœud 1 -> In
  Nœud 2 -> Acc
  Paiement SPE = (2, 1)  (Entrant, Incumbent)

Analyse menace (Fight) : au nœud incumbent (joueur 1),
  Acc = 1, Fight = -1  -> Accommoder domine strictement.
  => 'Fight' est une menace NON CRÉDIBLE (hors-chemin SPE).


### Interprétation des résultats

Le SPE est **(In, Acc)** avec paiement **(2, 1)**. Le profil `(Out, Fight)` — où l'entrant reste dehors par peur de la menace — est un **équilibre de Nash** (l'entrant sort car il croit à Fight ; l'incumbent annonce Fight, jamais testé car l'entrant sort). Mais ce n'est **pas un SPE** : si l'entrant entrait quand même, l'incumbent rationnel accommoderait — Accommodate domine strictement Fight à ce nœud (1 > −1). C'est le cœur de la distinction Nash vs SPE.

Le nombre de sous-jeux = 2 (nœuds 1 et 2) : la racine et le nœud incumbent.


In [4]:
// Pont vers Gambit (.efg) : valider le SPE avec un solveur externe (lib-vs-lib, #10382).
// Memes vehicule et discipline que GT-4 (.nfg) : Process.Start CLI, PAS de NuGet,
// pour respecter la GPL-2.0 de Gambit. Le pont .efg (forme extensive) est la suite
// naturelle du pont .nfg (forme normale) : il resout le SPE directement sur l'arbre.
using System.Diagnostics;
using System.Globalization;
using System.IO;

// Serialise un ExtensiveFormGame au format .efg (EFG 2, forme extensive Gambit).
// Parcours DFS preorder (l'ordre attendu par le parseur EFG). Information parfaite :
// chaque noeud de decision est son propre ensemble d'information (infoset singleton).
public static string ToEfg(ExtensiveFormGame game)
{
    var sb = new System.Text.StringBuilder();
    sb.Append($"EFG 2 R \"Extensive game\" {{ ");
    for (int p = 0; p < game.NPlayers; p++) sb.Append($"\"{game.PlayerNames[p]}\" ");
    sb.AppendLine("}");
    sb.AppendLine("\"\"");
    sb.AppendLine();
    var isetCount = new int[game.NPlayers + 1];  // infosets par joueur (1-indexe)
    int outcome = 0;
    void Emit(GameNode n)
    {
        if (n.IsTerminal)
        {
            outcome++;
            var pays = n.Payoffs!;
            var payStr = string.Join(", ", pays.Select(v => v.ToString("0.0##############", CultureInfo.InvariantCulture)));
            sb.AppendLine($"t \"\" {outcome} \"o{outcome}\" {{ {payStr} }}");
        }
        else
        {
            int pl = n.Player + 1;  // 0-indexe (GameNode) -> 1-indexe (EFG)
            isetCount[pl]++;
            var acts = string.Join(" ", n.Actions.Select(a => $"\"{a}\""));
            sb.AppendLine($"p \"\" {pl} {isetCount[pl]} \"\" {{ {acts} }} 0");
            foreach (var c in n.Children) Emit(c);
        }
    }
    Emit(game.Root);
    return sb.ToString();
}

// Cherche l'executable Gambit (.efg) : GAMBIT_HOME d'abord, puis installs
// Windows connues (utilisateur local, Program Files), puis emplacements Unix
// (Linux /usr/bin et /usr/local/bin ; macOS homebrew). Sur Unix les binaires
// Gambit n'ont pas de suffixe .exe. Pattern candidats de GT-5, etendu (#10643).
public static string FindGambitEfg(string solver)
{
    bool win = OperatingSystem.IsWindows();
    string exe = win ? solver + ".exe" : solver;
    var candidates = new List<string>();
    var home = Environment.GetEnvironmentVariable("GAMBIT_HOME");
    if (!string.IsNullOrEmpty(home)) candidates.Add(Path.Combine(home, exe));
    if (win)
    {
        candidates.Add(Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.LocalApplicationData), "Gambit", "Gambit", exe));
        candidates.Add($@"C:\Program Files (x86)\Gambit\{exe}");
        candidates.Add($@"C:\Program Files\Gambit\{exe}");
    }
    else
    {
        candidates.Add("/usr/bin/" + exe);
        candidates.Add("/usr/local/bin/" + exe);
        candidates.Add("/opt/homebrew/bin/" + exe);
    }
    foreach (var c in candidates)
        if (File.Exists(c)) return c;
    throw new FileNotFoundException(
        $"Binaire Gambit introuvable : {solver}. Cherche (GAMBIT_HOME, installs locales) :\n  - " +
        string.Join("\n  - ", candidates) +
        "\nInstaller Gambit (http://www.gambit-project.org/) ou definir GAMBIT_HOME.");
}

// Invoque un solveur Gambit sur le .efg, retourne les profils (NE behavioral ou strategique).
public static List<double[]> RunGambitEfg(ExtensiveFormGame game, string solver, bool strategic = false)
{
    string exe = FindGambitEfg(solver);
    string tmp = Path.GetTempFileName();
    File.WriteAllText(tmp, ToEfg(game));
    // gambit-enumpure (NE purs, strategies entieres) rejette -d ; -S = forme strategique.
    // - strategic=true            -> " -S"  (conversion vers la forme strategique)
    // - enumpure, strategic=false -> ""     (forme agent : enumeration SUR l'arbre, sans conversion)
    // - lcp, strategic=false      -> " -d 6" (decimales pour les probas comportementales)
    string sFlag = strategic ? " -S" : (solver.Contains("enumpure") ? "" : " -d 6");
    var psi = new ProcessStartInfo
    {
        FileName = exe,
        Arguments = $"-q{sFlag} \"{tmp}\"",
        RedirectStandardOutput = true, RedirectStandardError = true,
        UseShellExecute = false, CreateNoWindow = true,
    };
    using var proc = Process.Start(psi);
    string stdout = proc.StandardOutput.ReadToEnd();
    proc.WaitForExit();
    File.Delete(tmp);
    var result = new List<double[]>();
    foreach (var line in stdout.Split('\n', StringSplitOptions.RemoveEmptyEntries))
        if (line.StartsWith("NE,"))
        {
            var v = line.Substring(3).Split(',')
                .Select(s => double.Parse(s, CultureInfo.InvariantCulture)).ToArray();
            result.Add(v);
        }
    return result;
}

// --- Validation 1 : SPE behavioral (forme extensive, pivotage lcp) ---
var speBehav = RunGambitEfg(entryGame, "gambit-lcp");
$"gambit-lcp (forme extensive, pivotage) : {speBehav.Count} equilibre(s)".Display();
foreach (var eq in speBehav)
    $"  Probas (Out, In, Acc, Fight) = [{string.Join(", ", eq.Select(v => v.ToString("F4", CultureInfo.InvariantCulture)))}]".Display();
// Cross-validation : le backward induction (cellule precedente) a donne SPE = (In, Acc), payoff (2, 1).
string efgVerdict = (speBehav.Count >= 1
    && Math.Abs(speBehav[0][0]) < 1e-3 && Math.Abs(speBehav[0][1] - 1.0) < 1e-3
    && Math.Abs(speBehav[0][2] - 1.0) < 1e-3 && Math.Abs(speBehav[0][3]) < 1e-3)
    ? "OK -- gambit-lcp confirme le SPE (In, Acc) du backward induction"
    : "MISMATCH";
$"Cross-validation vs backward induction : {efgVerdict}".Display();

// --- Prong-B : (Out, Fight) est-il un Nash de l'arbre lui-meme ? ---
// gambit-enumpure SANS -S enumere les NE purs directement en forme agent (comportementale),
// SANS conversion vers la forme strategique. Contre-preuve decisive : si (Out, Fight) y
// apparait, ce n'est pas un artefact de la conversion en matrice -- c'est un Nash de la
// forme extensive elle-meme (le noeud incumbent n'etant jamais atteint, Fight y est une
// meilleure reponse faible).
var neAgent = RunGambitEfg(entryGame, "gambit-enumpure", strategic: false);
$"\ngambit-enumpure (forme agent, SUR l'arbre) : {neAgent.Count} NE purs".Display();
string[] actionLabels = { "Out", "In", "Acc", "Fight" };
foreach (var eq in neAgent)
{
    var played = new List<string>();
    for (int i = 0; i < eq.Length; i++) if (eq[i] > 0.5) played.Add(actionLabels[i]);
    bool isPerfect = played.Contains("In") && played.Contains("Acc");
    string tag = isPerfect ? "  <- SPE (sous-jeu-parfait)" : "  <- Nash NON sous-jeu-parfait (menace non credible hors-chemin)";
    $"  ({string.Join(", ", played)}){tag}".Display();
}

// Meme enumeration apres conversion strategique (-S) : la matrice rend le meme ensemble.
var neStrategic = RunGambitEfg(entryGame, "gambit-enumpure", strategic: true);
$"\ngambit-enumpure -S (forme strategique equivalente) : {neStrategic.Count} NE purs (idem -> la conversion ne fabrique rien)".Display();

// --- Bilan : aucun solveur ne filtre les menaces non credibles ---
// L'ecart 1-vs-2 oppose UN equilibre trouve par pivotage (lcp s'arrete au premier point
// complementaire) a TOUS les equilibres enumeres (enumpure). La forme extensive ne filtre
// rien : (Out, Fight) est un Nash de l'arbre. La perfection en sous-jeux est un raffinement
// qu'on IMPOSE (le backward induction de la cellule precedente), pas un don du solveur.
$"\nBilan : gambit-lcp (pivotage) = {speBehav.Count} eq ; gambit-enumpure (enumeration) = {neAgent.Count} Nash.".Display();
"La perfection en sous-jeux est un raffinement qu'on impose (backward induction), pas un filtrage du solveur.".Display();

gambit-lcp (forme extensive, pivotage) : 1 equilibre(s)

  Probas (Out, In, Acc, Fight) = [0.0000, 1.0000, 1.0000, 0.0000]

Cross-validation vs backward induction : OK -- gambit-lcp confirme le SPE (In, Acc) du backward induction


gambit-enumpure (forme agent, SUR l'arbre) : 2 NE purs

  (In, Acc)  <- SPE (sous-jeu-parfait)

  (Out, Fight)  <- Nash NON sous-jeu-parfait (menace non credible hors-chemin)


gambit-enumpure -S (forme strategique equivalente) : 2 NE purs (idem -> la conversion ne fabrique rien)


Bilan : gambit-lcp (pivotage) = 1 eq ; gambit-enumpure (enumeration) = 2 Nash.

La perfection en sous-jeux est un raffinement qu'on impose (backward induction), pas un filtrage du solveur.

**Lecture** : La validation croisée avec gambit-lcp confirme le SPE (In,Acc) trouvé par backward induction. Le solveur externe gambit-enumpure identifie deux équilibres de Nash purs : (In,Acc) qui est SPE, et (Out,Fight) qui n'est pas sous-jeu-parfait car il repose sur la menace non crédible 'Fight'. Ceci illustre que la perfection en sous-jeux est un raffinement plus strict que l'équilibre de Nash.
**Pourquoi une validation croisée (lib-vs-lib, #10382)** : deux implémentations indépendantes qui atteignent le même équilibre par des voies différentes (récursion maison sur l'arbre d'un côté, pivotage de complémentarité linéaire de l'autre) se corrigent mutuellement d'une erreur d'implémentation. La leçon fine ici : `gambit-lcp` renvoie **un seul** équilibre non parce qu'il filtrerait — mais parce que le pivotage s'arrête au **premier** point complémentaire trouvé. Ne jamais lire « le solveur n'en renvoie qu'un » comme « il n'en existe qu'un » : c'est la sortie de l'algorithme qui est tronquée, pas le jeu.

### Pourquoi les deux solveurs ne renvoient pas le même nombre d'équilibres

Les deux solveurs Gambit ci-dessus travaillent sur **le même arbre de jeu**, mais ne renvoient pas le même ensemble d'équilibres :

- **`gambit-lcp`** (sans `-S`) résout le jeu **directement sur la forme extensive** (forme séquentielle de Koller–Megiddo–von Stengel) et calcule des équilibres en **stratégies comportementales** (une probabilité par action à chaque nœud de décision). Il n'en renvoie qu'**un seul** : l'algorithme de complémentarité linéaire s'arrête au **premier point complémentaire atteint par pivotage**. Ici `(In, Acc)` — qui *se trouve être* le SPE, mais `lcp` ne l'a pas choisi pour sa perfection.
- **`gambit-enumpure -S`** convertit l'arbre en **forme stratégique** (la matrice des plans d'action contingents) puis **énumère tous** les équilibres de Nash en stratégies pures : ici 2, dont `(Out, Fight)`.

**Le point à retenir** : `(Out, Fight)` est un équilibre de Nash **de la forme extensive elle-même**, pas un artefact de la conversion vers la forme stratégique. Le nœud de l'incumbent n'étant jamais atteint (l'entrant joue `Out`), `Fight` y est une meilleure réponse *faible* — tout écart est sans conséquence. Et l'entrant obtient 1 avec `Out`, contre −1 s'il déviait vers `In` (où l'incumbent accommode). Aucun des deux ne gagne à dévier : c'est un Nash.

**Aucun solveur ne filtre les menaces non crédibles** : la perfection en sous-jeux est un **raffinement qu'on impose** — ici par le `BackwardInduction` de la cellule précédente, qui exige l'optimalité à *chaque* sous-jeu, y compris celui que `(Out, Fight)` n'atteint jamais (et où `Acc` bat `Fight`). L'écart 1-vs-2 oppose donc *un* équilibre trouvé par pivotage à *tous* les équilibres énumérés — non la forme extensive à la forme stratégique.

C'est précisément la leçon de Selten (1965) : l'équilibre de Nash ne suffit pas en présence de menaces hors-chemin ; il faut un **raffinement** (le SPE) pour les écarter. Le pont `.efg` le rend mécanique — le solveur externe (Gambit, la référence du domaine pour la théorie des jeux) confirme le verdict du backward induction from-scratch écrit plus haut, et l'option `-S` rend visibles les Nash non-parfaits que ce backward induction élimine.

> **Parité lib-vs-lib et axes de pont** (#10382) : le pont C# ci-dessus invoque le **vrai solveur Gambit** (moteur C++) via sa ligne de commande (`Process.Start` sur `gambit-lcp.exe`) — l'axe « binaire CLI » de la taxonomie de parité. Gambit expose aussi un binding Python (`pygambit`) ; un chemin alternatif pour le côté C# serait l'interopérabilité CLR `PythonNet → pygambit → Gambit C++` (axe « PythonNet », déjà certifié `SOTA-OK` ailleurs dans le dépôt). Le choix de la CLI ici suit la même discipline que le pont `.nfg` de GT-4 : zéro dépendance runtime Python, conformité GPL-2.0 préservée (pas de liaison NuGet vers la bibliothèque), invocation idiomatique de .NET. À noter : le jumeau Python de ce notebook reste **from-scratch** (il n'invoque pas encore `pygambit`) — le pont C# ouvre donc le déploiement lib-vs-lib côté .NET, et le Python pourrait suivre la même voie.

## 2. Menaces et promesses crédibles

### 2.1 Menace non crédible
On parle de **menace non crédible** quand un joueur annonce une action punitive qui **n'est pas dans son intérêt** une fois le nœud atteint. Le SPE l'élimine automatiquement (rationalité à tous les nœuds).

### 2.2 Rendre une menace crédible
Pour rendre une menace crédible, un joueur peut **s'engager** (brûler ses ponts, signer un contrat, déléguer à un agent). En modifiant l'arbre (par exemple en retirant l'option Accommoder), la menace devient rationnelle au nœud atteint, donc **survit au SPE**.
**L'engagement transfère de la valeur, il n'en crée pas** : comparez les deux SPE — (In, Acc) à (2, 1) avant, (Out) à (1, 2) après l'engagement. L'entrant perd exactement **1** (2 → 1), l'incumbent gagne exactement **1** (1 → 2). La somme des paiements ne change pas d'un point : l'engagement ne rend pas le jeu plus efficace, il **déplace l'issue** vers le joueur qui a su se lier les mains en premier. C'est la lecture stratégique de Schelling : la liberté de choisir plus tard peut valoir *moins* que l'irréversibilité maintenant.

In [5]:
// Rendre la menace credible : l'incumbent s'engage publiquement a Combattre.
// On modifie l'arbre : l'incumbent n'a plus que 'Fight' (Acc retire).
var nAcc2 = new GameNode(4, -1, new double[]{ 2.0, 1.0 });
var nFight2 = new GameNode(5, -1, new double[]{ -1.0, 3.0 });  // Fight desormais rentable (engagement)
var incumbentCommit = new GameNode(2, 1);
incumbentCommit.AddChild("Acc", nAcc2).AddChild("Fight", nFight2);
var entrantCommit = new GameNode(1, 0);
entrantCommit.AddChild("Out", new GameNode(3,-1,new double[]{1.0,2.0}))
            .AddChild("In", incumbentCommit);

var (spe2, path2) = BackwardInduction(entrantCommit);

var sb = new System.Text.StringBuilder();
sb.AppendLine("Variante 'engagement' : Fight rapporte desormais 3 a l'incumbent (engagement credible).");
sb.AppendLine($"  Au nœud incumbent : Acc = {nAcc2.Payoffs![1]}, Fight = {nFight2.Payoffs![1]} -> Fight domine.");
sb.AppendLine("  => Menace rendue credible, le SPE bascule :");
foreach (var (nid, act) in path2) sb.AppendLine($"    Nœud {nid} -> {act}");
sb.AppendLine($"  Paiement SPE = ({spe2[0]}, {spe2[1]})");
sb.AppendLine();
sb.AppendLine("Leçon : changer la structure de paiements (engagement) transforme une menace");
sb.AppendLine("non credible en credible, et modifie l'issue du jeu (l'entrant prefere sortir).");
Show(sb.ToString());


Variante 'engagement' : Fight rapporte desormais 3 a l'incumbent (engagement credible).
  Au nœud incumbent : Acc = 1, Fight = 3 -> Fight domine.
  => Menace rendue credible, le SPE bascule :
    Nœud 1 -> Out
  Paiement SPE = (1, 2)

Leçon : changer la structure de paiements (engagement) transforme une menace
non credible en credible, et modifie l'issue du jeu (l'entrant prefere sortir).


**Lecture** : En modifiant le paiement de Fight à 3 pour l'incumbent (engagement crédible), Fight domine maintenant Accommodate au nœud incumbent, ce qui fait basculer le SPE vers (Out,Out) avec paiement (1,2). Ceci démontre comment un engagement visible peut rendre une menace crédible et changer l'issue du jeu.

### Le paradoxe de l'engagement
**Paradoxe** : se lier les mains (retirer une option, s'engager à une action coûteuse ex ante) peut **augmenter** le paiement d'un joueur. C'est le principe de la **dissuasion par engagement** (Schelling) : la crédibilité d'une menace vient de l'impossibilité de revenir en arrière.


## 3. Induction avant (Forward Induction)

### 3.1 Principe
L'induction avant interprète une action passée à partir de la rationalité supposée du joueur et de croyances explicites. Elle ne suit pas du seul calcul des équilibres du sous-jeu.

### 3.2 Exemple : Chasse au Cerf avec option extérieure
J1 choisit `Out`, qui lui donne 3,5, ou `In`, qui conduit au sous-jeu : Stag/Stag donne (4,4), Stag/Hare (0,3), Hare/Stag (3,0) et Hare/Hare (3,3).

Le meilleur gain de J1 en jouant Hare est 3, inférieur à `Out`, tandis que Stag peut donner 4. Sous l'hypothèse qu'entrer vise un gain strictement supérieur à `Out`, J2 peut inférer Stag puis y répondre par Stag. Ce raisonnement conditionnel ne calcule ni les croyances ni tous les équilibres du jeu complet. Le témoin disparaît pour `Out = 2` (les deux actions peuvent le dépasser) ou `Out = 4` (aucune ne le dépasse strictement).


In [6]:
// Forward induction sur Stag Hunt + option exterieure (Cho-Kreps).
// Matrice du sous-jeu Stag Hunt (lignes = P1, colonnes = P2) : [Stag, Hare]
double[,] stagHunt = {
    { 4.0, 0.0 },   // P1 Stag : (vs Stag=4, vs Hare=0)
    { 3.0, 3.0 }    // P1 Hare : (vs Stag=3, vs Hare=3)
};
// P2 (colonne) paiements :
double[,] stagHuntP2 = {
    { 4.0, 3.0 },   // P2 vs P1=Stag : (Stag=4, Hare=3)
    { 0.0, 3.0 }    // P2 vs P1=Hare : (Stag=0, Hare=3)
};
string[] actions = { "Stag", "Hare" };
double outPayoff = 3.5;  // option exterieure garantie

// Equilibres de Nash du sous-jeu (strategies pures) : identifier les best-responses croises.
var bestP1 = new List<int>();   // meilleure reponse de P1 a chaque action de P2
var bestP2 = new List<int>();
for (int j = 0; j < 2; j++)
{
    int bi = 0;
    for (int i = 1; i < 2; i++) if (stagHunt[i, j] > stagHunt[bi, j]) bi = i;
    bestP1.Add(bi);
}
for (int i = 0; i < 2; i++)
{
    int bj = 0;
    for (int j = 1; j < 2; j++) if (stagHuntP2[i, j] > stagHuntP2[i, bj]) bj = j;
    bestP2.Add(bj);
}
var nashPure = new List<(int i, int j)>();
for (int i = 0; i < 2; i++)
    for (int j = 0; j < 2; j++)
        if (bestP1[j] == i && bestP2[i] == j)
            nashPure.Add((i, j));

var sb = new System.Text.StringBuilder();
sb.AppendLine($"Stag Hunt + option exterieure (Out = {outPayoff})");
sb.AppendLine("Equilibres de Nash purs du sous-jeu Stag Hunt :");
foreach (var (i, j) in nashPure)
    sb.AppendLine($"  ({actions[i]}, {actions[j]}) -> paiement ({stagHunt[i,j]}, {stagHuntP2[i,j]})");
sb.AppendLine();
double hareBest = Math.Max(stagHunt[1,0], stagHunt[1,1]);
double stagBest = Math.Max(stagHunt[0,0], stagHunt[0,1]);
if (!(hareBest < outPayoff && outPayoff < stagBest))
    throw new InvalidOperationException("Les inegalites du temoin ne sont pas satisfaites.");
sb.AppendLine($"Out={outPayoff} ; maximum Stag={stagBest} ; maximum Hare={hareBest}");
sb.AppendLine($"Stag peut depasser Out : {stagBest > outPayoff} ; Hare : {hareBest > outPayoff}");
sb.AppendLine("Sous rationalite stricte et croyances appropriees, In peut signaler Stag.");
sb.AppendLine("Les equilibres du jeu complet ne sont pas enumeres ici.");
Show(sb.ToString());


Stag Hunt + option exterieure (Out = 3.5)
Equilibres de Nash purs du sous-jeu Stag Hunt :
  (Stag, Stag) -> paiement (4, 4)
  (Hare, Hare) -> paiement (3, 3)

Out=3.5 ; maximum Stag=4 ; maximum Hare=3
Stag peut depasser Out : True ; Hare : False
Sous rationalite stricte et croyances appropriees, In peut signaler Stag.
Les equilibres du jeu complet ne sont pas enumeres ici.


**Lecture.** Les deux équilibres purs du sous-jeu restent (Stag, Stag) et (Hare, Hare). Le gain maximal de Hare est inférieur à `Out`, celui de Stag peut le dépasser. Sous l'hypothèse que J1 choisit `In` pour obtenir strictement plus que `Out`, J2 peut inférer Stag et y répondre par Stag. Le calcul établit les inégalités ; les croyances et les équilibres du jeu complet ne sont pas calculés ici.


### Pourquoi l'induction avant est puissante
L'induction avant interprète les actions passées sous des hypothèses de rationalité et de croyances. Dans cet exemple, renoncer à l'option sûre peut signaler l'intention de jouer Stag : seule cette action peut rapporter strictement davantage que l'option extérieure. Le calcul vérifie cette comparaison ; il ne détermine ni les croyances ni une sélection générale parmi les SPE. Les jeux de signal (Spence) développent le rôle des actions observées et des croyances en information incomplète.


## 4. Perfection en mains tremblantes (Trembling-hand perfection)

### 4.1 Motivation
Le SPE exige un équilibre dans chaque sous-jeu, y compris hors du chemin joué ; il n'autorise donc pas une action arbitraire dans un sous-jeu non atteint. Les tremblements ajoutent un test de robustesse aux erreurs, distinct de cette exigence. La **perfection en mains tremblantes** (Selten 1975) exige que l'équilibre soit robuste à de **petites erreurs** (tremblements) : chaque joueur joue chaque action avec une probabilité ε > 0, et l'équilibre est la limite quand ε → 0.

### 4.2 Définition
Un profil est **trembling-hand parfait** s'il est la limite d'une suite d'équilibres ε-perturbés où **toutes** les actions sont jouées avec probabilité positive.

### 4.3 Conséquences
La perfection élimine les équilibres qui nécessitent qu'un joueur soit **certain** qu'un nœud hors-chemin ne sera jamais atteint.
**L'arithmétique du tremblement** : avec ε = 0,1, l'entrant obtient In→Acc dans 90 % des cas (paiement 2) et bascule vers Out dans 10 % (paiement 1) — espérance **0,9 × 2 + 0,1 × 1 = 1,900**, toujours au-dessus du Out pur (1). Notez *pourquoi* la dominance survit : elle est **locale au nœud** de l'incumbent (Acc 1 vs Fight −1 ne dépend pas de ce que fait l'entrant), donc aucun tremblement en amont ne peut l'affecter. La perfection en mains tremblantes ne teste pas la robustesse d'un calcul global — elle teste si chaque décision d'équilibre reste la meilleure *quand les nœuds hors-chemin redeviennent possibles*.

In [7]:
// Trembling-hand : on perturbe le jeu d'entree par epsilon (l'entrant peut 'trembler').
// Entrant joue In avec (1-eps), Out avec eps. Au nœud incumbent, on calcule le paiement
// esperé de Acc vs Fight, pour verifier la robustesse du SPE sous tremblement.
double eps = 0.10;   // 10% de tremblement
// Au nœud incumbent (atteint avec proba 1-eps), les paiements Acc/Fight sont inchanges
// (le tremblement est sur le choix de l'entrant, pas sur l'incumbent).
double accIncumbent = 1.0;     // paiement si Accommodate
double fightIncumbent = -1.0;  // paiement si Fight
bool accDominates = accIncumbent > fightIncumbent;

// Esperance pour l'entrant sous (In->Acc) avec tremblement Out :
double entrantInAcc = (1 - eps) * 2.0 + eps * 1.0;   // In(2) w.p. 1-eps, Out(1) w.p. eps
double entrantOut = 1.0;
bool inPreferred = entrantInAcc > entrantOut;

var sb = new System.Text.StringBuilder();
sb.AppendLine($"Trembling-hand perfection (eps = {eps}) sur le jeu d'entree");
sb.AppendLine($"  Au nœud incumbent : Acc = {accIncumbent}, Fight = {fightIncumbent}");
sb.AppendLine($"    -> Accommodate domine : {accDominates} (robuste sous tremblement, le dominant reste dominant).");
sb.AppendLine($"  Esperance entrant (In->Acc) avec tremblement Out = {entrantInAcc:F3}");
sb.AppendLine($"    vs Out pur = {entrantOut}  -> In preferable : {inPreferred}");
sb.AppendLine();
sb.AppendLine("Verdict : le SPE (In, Acc) est trembling-hand parfait. Meme avec 10% de chance");
sb.AppendLine("de 'trembler' vers Out, Accommodate reste strictement dominant pour l'incumbent,");
sb.AppendLine("et In reste preferable pour l'entrant. L'equilibre ne s'appuie sur aucune certitude");
sb.AppendLine("qu'un nœud hors-chemin serait evite.");
Show(sb.ToString());


Trembling-hand perfection (eps = 0.1) sur le jeu d'entree
  Au nœud incumbent : Acc = 1, Fight = -1
    -> Accommodate domine : True (robuste sous tremblement, le dominant reste dominant).
  Esperance entrant (In->Acc) avec tremblement Out = 1.900
    vs Out pur = 1  -> In preferable : True

Verdict : le SPE (In, Acc) est trembling-hand parfait. Meme avec 10% de chance
de 'trembler' vers Out, Accommodate reste strictement dominant pour l'incumbent,
et In reste preferable pour l'entrant. L'equilibre ne s'appuie sur aucune certitude
qu'un nœud hors-chemin serait evite.


### Interprétation des résultats numériques
La perfection en mains tremblantes confirme le SPE **(In, Acc)** : parce que Accommodate **domine strictement** Fight, aucun tremblement ne renverse la préférence de l'incumbent. Certains équilibres sont éliminés lorsque les perturbations détruisent leur propriété de meilleure réponse. Une indifférence, à elle seule, ne prouve pas l'absence de perfection : il faut vérifier les meilleures réponses dans les jeux perturbés.

La lecture de la sortie le confirme des deux côtés : avec ε = 0.1 de tremblement vers Out, `In` reste préférable pour l'entrant (espérance `1,900` contre `1` pour Out pur), et côté incumbent `Acc = 1` domine `Fight = -1`.


## 5. Application : le jeu du « Burn Money »

Dans la bataille des sexes, J1 préfère Opéra et J2 préfère Football. Supposons que la continuation sans brûlage soit (Football, Football), qui donne 2 à J1. Avec un coût de 0,5, viser Opéra après brûlage peut donner 2,5 ; viser Football donne au mieux 1,5. Sous cette hypothèse et une rationalité stricte, seul Opéra peut justifier le signal. Le programme vérifie ces inégalités, sans déterminer toutes les croyances ni tous les équilibres.


In [8]:
// Burn money : modele simplifie en deux etapes.
// Etape 1 : P1 choisit Burn (cout c) ou Not.
// Etape 2 : jeu de coordination (Battle of Sexes simplifie).
//   Hypothese examinee : apres Burn, P2 infere l'action preferee de P1.
// Matrice BoS : [Opera, Football] -- P1 prefere Opera, P2 prefere Football.
double[,] bosP1 = { { 3.0, 0.0 }, { 0.0, 2.0 } };  // P1 (Opera, Football) vs P2 (Opera, Football)
double[,] bosP2 = { { 2.0, 0.0 }, { 0.0, 3.0 } };
double c = 0.5;   // cout du burn

// Comparaison conditionnelle à (Football, Football) sans brûlage.
double p1OperaBurn = Math.Max(bosP1[0,0], bosP1[0,1]) - c;
double p1FootballBurn = Math.Max(bosP1[1,0], bosP1[1,1]) - c;
double p1FootballNoBurn = bosP1[1,1];
if (!(p1OperaBurn > p1FootballNoBurn && p1FootballNoBurn > p1FootballBurn))
    throw new InvalidOperationException("Les inegalites du temoin ne sont pas satisfaites.");
var sb = new System.Text.StringBuilder();
sb.AppendLine($"Burn={c} ; maximum Opera={p1OperaBurn} ; maximum Football={p1FootballBurn} ; reference={p1FootballNoBurn}");
sb.AppendLine($"Opera peut depasser la reference : {p1OperaBurn > p1FootballNoBurn} ; Football : {p1FootballBurn > p1FootballNoBurn}");
sb.AppendLine("Conclusion conditionnelle a (Football,Football) sans burn et aux croyances.");
Show(sb.ToString());


Burn=0.5 ; maximum Opera=2.5 ; maximum Football=1.5 ; reference=2
Opera peut depasser la reference : True ; Football : False
Conclusion conditionnelle a (Football,Football) sans burn et aux croyances.


### Portée du signal coûteux

La comparaison prend (Football, Football) comme continuation sans brûlage. Sous cette hypothèse, le coût rend Football non profitable après le signal, mais Opéra peut dépasser le gain de référence. Les croyances restent à préciser : ce calcul ne démontre pas une sélection universelle.


## 6. Comparaison des raffinements

Les notions mobilisent des exigences distinctes : Nash compare les déviations, SPE impose un équilibre dans chaque sous-jeu, la perfection étudie la robustesse aux petites erreurs et l'induction avant considère les intentions compatibles avec les actions observées. L'induction avant ne constitue pas un maillon d'une chaîne universelle d'inclusions.

### Perfection extensive, forme normale et SPE

Dans un **jeu fini à mémoire parfaite**, la perfection en mains tremblantes de la **forme extensive** impose des tremblements à chaque ensemble d'information. Tout équilibre ainsi parfait est un **équilibre séquentiel**, donc un **SPE**. La perfection de la **forme normale** perturbe, elle, les plans d'action complets : elle ne garantit pas la perfection en sous-jeux. Il faut donc préciser la forme du jeu avant de comparer ces notions.

La réciproque « SPE ⇒ perfection extensive » est fausse, même à information parfaite en présence d'égalités de gains. Considérons J1 : **Out → (1, 0)**, ou **In**, puis J2 : **x → (1, 1)** ou **y → (0, 0)**. Le profil **(In, x)** est un SPE : J2 préfère strictement x et J1 est indifférent entre Out et In. Mais si J2 tremble vers y avec une probabilité ε > 0, In rapporte **1 − ε < 1**, contre 1 pour Out. In ne peut donc pas être l'action limite d'une meilleure réponse dans les jeux perturbés : ce SPE n'est pas parfait en forme extensive.

Ce contre-exemple théorique est vérifié à la main ; il ne correspond pas à une sortie des cellules exécutées. Les calculs du carnet illustrent des cas particuliers, sans calculer l'ensemble des équilibres parfaits ou des croyances séquentielles.

Pour ce cadre fini à mémoire parfaite : **perfection extensive ⇒ équilibre séquentiel ⇒ SPE ⇒ Nash**. Le tableau calculé ci-dessous porte sur les exemples du carnet, pas sur une démonstration de ces inclusions générales.


In [9]:
// Synthese : hierarchie des raffinements sur nos exemples.
var sb = new System.Text.StringBuilder();
sb.AppendLine("Raffinements : resultats calcules et limites");
sb.AppendLine(new string('-', 60));
sb.AppendLine($"{"Raffinement",-22}{"Jeu d'entree",-20}{"Stag Hunt+Out",-18}");
sb.AppendLine(new string('-', 60));
sb.AppendLine($"{"Nash",-22}{"(In,Acc),(Out,F)",-20}{"sous-jeu : 3 eq.",-18}");
sb.AppendLine($"{"SPE",-22}{"(In,Acc) seul",-20}{"non enumeres",-18}");
sb.AppendLine($"{"Trembling-hand",-22}{"(In,Acc) robuste",-20}{"non calcule",-18}");
sb.AppendLine($"{"Forward induction",-22}{"(In,Acc)",-20}{"Stag conditionnel",-18}");
sb.AppendLine(new string('-', 60));
sb.AppendLine();
sb.AppendLine("Les lignes resument des exigences distinctes ; elles ne prouvent");
sb.AppendLine("aucune relation generale d'inclusion entre ces raffinements.");
Show(sb.ToString());


Raffinements : resultats calcules et limites
------------------------------------------------------------
Raffinement           Jeu d'entree        Stag Hunt+Out     
------------------------------------------------------------
Nash                  (In,Acc),(Out,F)    sous-jeu : 3 eq.  
SPE                   (In,Acc) seul       non enumeres      
Trembling-hand        (In,Acc) robuste    non calcule       
Forward induction     (In,Acc)            Stag conditionnel 
------------------------------------------------------------

Les lignes resument des exigences distinctes ; elles ne prouvent
aucune relation generale d'inclusion entre ces raffinements.


### Interprétation du tableau comparatif
- Sur le jeu d'entrée, SPE écarte la menace non crédible.
- Dans la chasse au cerf, la comparaison des gains ne permet d'inférer Stag que sous les hypothèses de rationalité et de croyances indiquées ; les raffinements du jeu complet ne sont pas énumérés.
- Le choix du raffinement dépend du contexte ; le tableau ne remplace pas une vérification des stratégies complètes.


## 7. Exercices

> Les exercices ci-dessous sont des **stubs** à compléter. Ils utilisent `pass`/`return null` (jamais `throw`) : le notebook s'exécute de bout en bout même non complété (règle C.1).

**Cadrage** : les cinq exercices se traitent avec la seule API de la cellule 3 — construire un arbre (`GameNode`), le passer à `BackwardInduction`. Les stubs s'exécutent sans erreur non complétés (règle C.1 : `return null`, jamais `throw`).
### Exercice 1 — Beer-Quiche Game
Le jeu **Beer-Quiche** (Cho-Kreps 1987) : un joueur de type Fort ou Faible choisit de manger Bière ou Quiche ; l'autre joueur observe le choix (pas le type) et décide Combattre ou Ne pas combattre. Implémenter l'arbre et identifier l'équilibre de signal.

*Indice (Exercice 1)* : dans ce jeu de signal à deux types, la question n'est pas « quelle action » mais « quel **équilibre** » — séparateur (chaque type envoie un message distinct) ou pooling (les deux types envoient le même). Demandez-vous ce que l'observateur peut inférer d'un Quiche servi par un joueur Fort : rien, si l'équilibre est pooling ; tout, s'il est séparateur. Le paiement de la nature (probabilités des types) détermine lequel survit.


In [10]:
// Exercice 1 — Beer-Quiche (a completer).
#nullable enable
// TODO etudiant : construire l'arbre (nature choisit type Fort/Naible, P1 choisit Beer/Quiche,
// P2 observe l'action puis choisit Fight/NotFight) et identifier l'equilibre separateur vs melange.
// Indice : l'equilibre pooling (les deux types jouent Beer) survit si P2 croit "Fort" voyant Beer.
// Etape 1 : definir les nœuds et paiements.
// Etape 2 : appliquer BackwardInduction sur chaque sous-jeu d'information (ici, hors scope info imparfaite).
Console.WriteLine("Exercice 1 (Beer-Quiche) a completer.");


Exercice 1 (Beer-Quiche) a completer.


### Exercice 2 — Négociation avec options extérieures
Deux joueurs négocient (jeu séquentiel type Rubinstein) avec des options extérieures. Montrer comment une option extérieure crédible améliore le paiement d'un joueur.
*Indice* : la menace de quitter la table n'est crédible que si l'option extérieure est **réellement jouable** — comparez le SPE avec option extérieure purement déclarée (non jouable au nœud) et réellement accessible. La valeur d'une option extérieure n'est pas son montant, c'est **le paiement SPE qu'elle déclenche**.

In [11]:
// Exercice 2 — Negociation avec options exterieures (a completer).
#nullable enable
// TODO etudiant : modeliser un jeu de negotiation a 2 tours avec options exterieures.
// Indice : l'option exterieure borne le paiement minimum (outside option principle).
// Etape 1 : construire l'arbre a 2 tours. Etape 2 : backward induction.
Console.WriteLine("Exercice 2 (Negociation) a completer.");


Exercice 2 (Negociation) a completer.


### Exercice 3 — Engagement séquentiel
Un joueur peut s'engager séquentiellement (capacité de production, R&D) avant un jeu de marché. Implémenter l'arbre et mesurer la valeur de l'engagement.
*Indice* : mesurez la **valeur de l'engagement** comme la différence de paiement SPE avant/après (exactement la lecture §2 : +1 pour l'engagé, −1 pour l'autre). Un engagement qui ne change pas le SPE n'a aucune valeur stratégique — seulement un coût.

In [12]:
// Exercice 3 — Engagement sequentiel (a completer).
#nullable enable
// TODO etudiant : arbre a deux etages (engagement puis competition Cournot/Stackelberg).
// Indice : Stackelberg = engagement de capacite, le leader gagne plus qu'en Cournot simultane.
Console.WriteLine("Exercice 3 (Engagement) a completer.");


Exercice 3 (Engagement) a completer.


### Exercice 4 — Jeu d'entrée avec signal de prix
Un entrant signale sa qualité par un prix d'introduction bas (limit pricing). Implémenter la logique de signal.
*Indice* : le signal de prix est **inversé** par rapport à l'intuition — un prix *bas* signale la force (seul un entrant de haute qualité peut se permettre la marge sacrifiée), un prix haut signalerait la faiblesse. Construisez la matrice où le coût du signal bas est supportable pour le type haut et ruineux pour le type bas.

In [13]:
// Exercice 4 — Signal de prix (a completer).
#nullable enable
// TODO etudiant : modele de signal (qualite haute/basse) avec cout du signal croise.
// Indice : signal credible = cout plus faible pour haute qualite que pour basse (condition de Spence).
Console.WriteLine("Exercice 4 (Signal prix) a completer.");


Exercice 4 (Signal prix) a completer.


### Exercice 5 — Identification des SPE
Étant donné un arbre de jeu (fourni), énumérer **tous** les SPE (pas seulement celui de backward induction pur — gérer les égalités).
*Indice* : la récursion de la §1 choisit *une* action maximisante ; l'énumération complète exige de collecter **toutes** les actions atteignant le maximum à chaque nœud (égalités), puis de composer les chemins — c'est un produit cartésien de choix locaux, pas une deuxième recherche de maximum.

In [14]:
// Exercice 5 — Tous les SPE (a completer).
#nullable enable
// TODO etudiant : etendre BackwardInduction pour enumerer TOUS les SPE (en cas d'egalite de paiement),
// en retournant l'ensemble des profils optimaux a chaque nœud.
// Indice : au lieu d'un argmax unique, collecter tous les argmax a chaque nœud puis faire le produit cartesien.
Console.WriteLine("Exercice 5 (Tous les SPE) a completer.");


Exercice 5 (Tous les SPE) a completer.


## 8. Résumé

### Points clés
- **SPE** (backward induction) = équilibre de Nash dans **chaque** sous-jeu ; élimine les **menaces non crédibles**.
- **Menace crédible** = action punitive rationnelle au nœud atteint ; on peut la **créer** par engagement (changer l'arbre).
- **Trembling-hand perfection** = robustesse aux petites erreurs (ε).
- **Forward induction** = les actions passées **signalent** les intentions (logique Cho-Kreps).
- **Burn money** = signal coûteux auto-infligé ; sa portée dépend des continuations et des croyances.

### Hiérarchie
Ces raffinements imposent des exigences distinctes. Les calculs ci-dessus illustrent certains cas ; ils ne prouvent pas une relation générale d'équivalence ou d'inclusion entre eux.

### Prochaine étape
- [GameTheory-11 (Bayésien)](GameTheory-11-BayesianGames-CSharp.ipynb) : information incomplète.
- [GameTheory-12 (Réputation)](GameTheory-12-ReputationGames-CSharp.ipynb) : jeux répétés et signal.

***

**Twin C# .NET Interactive** du notebook Python `GameTheory-10-ForwardInduction-SPE-Python.ipynb`. Implémentation from-scratch (BCL .NET 9, 0 NuGet).
### Limites de ce jumeau (et prolongements)

Le solveur maison ne traite que l'**information parfaite** ; il ne démontre pas d'équivalence générale entre SPE et trembling-hand. Le pont `.efg` de la §1 illustre séparément le recours à Gambit. Les ensembles d'information de Beer-Quiche (Exercice 1) en sont la porte d'entrée naturelle. Prolongements : jeux répétés et réputation (une menace non crédible en une période peut le devenir sur l'horizon long), et le passage aux jeux de signal à la Spence où le coût du signal est continu.

**Jumeau Python** : `GameTheory-10-ForwardInduction-SPE-Python.ipynb` présente les mêmes exemples. Pour la chasse au cerf et le brûlage, la parité porte sur les matrices, l'option extérieure de 3,5, le coût de 0,5 et les comparaisons conditionnelles ; elle ne transforme pas celles-ci en sélection universelle d'équilibres.